In [ ]:
import pandas as pd

In [ ]:
tickets = pd.read_csv("tickets.csv")
tickets.head()

In [ ]:
tickets.shape


(11816, 21)

In [ ]:
tickets.columns

Index(['ticket_id', 'created_at', 'first_response_at', 'resolved_at', 'status',
       'channel', 'customer_id', 'order_id', 'product_sku', 'category',
       'priority', 'assigned_team', 'agent_id', 'transfers', 'csat_score',
       'refund_amount_inr', 'refund_reason_code', 'replacement_issued',
       'customer_message', 'agent_notes', 'source_system'],
      dtype='object')

In [ ]:
tickets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11816 entries, 0 to 11815
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ticket_id           11816 non-null  object 
 1   created_at          11816 non-null  object 
 2   first_response_at   11816 non-null  object 
 3   resolved_at         11192 non-null  object 
 4   status              11816 non-null  object 
 5   channel             11816 non-null  object 
 6   customer_id         11816 non-null  object 
 7   order_id            7795 non-null   object 
 8   product_sku         11816 non-null  object 
 9   category            11816 non-null  object 
 10  priority            11816 non-null  object 
 11  assigned_team       11816 non-null  object 
 12  agent_id            11816 non-null  object 
 13  transfers           11816 non-null  int64  
 14  csat_score          7339 non-null   float64
 15  refund_amount_inr   2095 non-null   float64
 16  refu

In [ ]:
tickets.isna().sum()

,0
ticket_id,0
created_at,0
first_response_at,0
resolved_at,624
status,0
channel,0
customer_id,0
order_id,4021
product_sku,0
category,0


In [ ]:
tickets["status"].value_counts()

,count
status,
resolved,10116
closed,1076
open,372
pending,252


In [ ]:
tickets.groupby("status")["resolved_at"].apply(lambda x: x.isna().sum())

,resolved_at
status,
closed,0
open,372
pending,252
resolved,0


In [ ]:
tickets[["created_at", "first_response_at", "resolved_at"]].head()

,created_at,first_response_at,resolved_at
0,2025-01-01 06:12,2025-01-01 06:20,2025-01-02 06:33
1,2025-01-01 09:01,2025-01-01 09:34,2025-01-01 09:45
2,2025-01-02 00:41,2025-01-02 03:41,2025-01-04 03:59
3,2025-01-02 02:21,2025-01-02 03:38,2025-01-12 04:37
4,2025-01-02 04:05,2025-01-02 04:38,2025-01-02 05:09


In [ ]:
tickets[["created_at", "first_response_at", "resolved_at"]].dtypes

,0
created_at,object
first_response_at,object
resolved_at,object


In [ ]:
time_cols = ["created_at", "first_response_at", "resolved_at"]

for col in time_cols:
    tickets[col] = pd.to_datetime(tickets[col], utc=True, errors="coerce")

In [ ]:
tickets[time_cols].isna().sum()


,0
created_at,0
first_response_at,0
resolved_at,624


In [ ]:
for col in time_cols:
    tickets[col] = tickets[col].dt.tz_convert("Asia/Kolkata")

In [ ]:
tickets[["created_at", "first_response_at"]].head()

,created_at,first_response_at
0,2025-01-01 11:42:00+05:30,2025-01-01 11:50:00+05:30
1,2025-01-01 14:31:00+05:30,2025-01-01 15:04:00+05:30
2,2025-01-02 06:11:00+05:30,2025-01-02 09:11:00+05:30
3,2025-01-02 07:51:00+05:30,2025-01-02 09:08:00+05:30
4,2025-01-02 09:35:00+05:30,2025-01-02 10:08:00+05:30


In [ ]:
tickets["response_minutes"] = (
    tickets["first_response_at"] - tickets["created_at"]
).dt.total_seconds() / 60

In [ ]:
tickets[["channel", "created_at", "first_response_at", "response_minutes"]].head()

,channel,created_at,first_response_at,response_minutes
0,voice,2025-01-01 11:42:00+05:30,2025-01-01 11:50:00+05:30,8.0
1,social,2025-01-01 14:31:00+05:30,2025-01-01 15:04:00+05:30,33.0
2,email,2025-01-02 06:11:00+05:30,2025-01-02 09:11:00+05:30,180.0
3,social,2025-01-02 07:51:00+05:30,2025-01-02 09:08:00+05:30,77.0
4,social,2025-01-02 09:35:00+05:30,2025-01-02 10:08:00+05:30,33.0


In [ ]:
sla_minutes = {
    "chat": 15,
    "voice": 120,
    "social": 240,
    "email": 480
}
tickets["sla_minutes"] = tickets["channel"].map(sla_minutes)
tickets[["channel", "response_minutes", "sla_minutes"]].head()

,channel,response_minutes,sla_minutes
0,voice,8.0,120
1,social,33.0,240
2,email,180.0,480
3,social,77.0,240
4,social,33.0,240


In [ ]:
tickets["sla_breach"] = (
    tickets["response_minutes"] > tickets["sla_minutes"]
)
tickets[["channel", "response_minutes", "sla_minutes", "sla_breach"]].head(10)
tickets["sla_breach"].value_counts()

,count
sla_breach,
False,9261
True,2555


In [ ]:
tickets["source_system"].value_counts()

,count
source_system,
helpdesk,8079
legacy_fd,3737


In [ ]:
tickets["ticket_id"].duplicated().sum()

np.int64(616)

In [ ]:
duplicate_tickets = tickets[
    tickets["ticket_id"].duplicated(keep=False)
]

duplicate_tickets[
    ["ticket_id", "source_system", "created_at", "channel", "agent_id"]
].sort_values("ticket_id").head(20)

,ticket_id,source_system,created_at,channel,agent_id
12,TK-240016,helpdesk,2025-01-03 09:08:00+05:30,voice,A3024
13,TK-240016,legacy_fd,2025-01-03 09:08:00+05:30,voice,A3024
15,TK-240018,helpdesk,2025-01-03 21:33:00+05:30,chat,A3029
16,TK-240018,legacy_fd,2025-01-03 21:33:00+05:30,chat,A3029
19,TK-240021,helpdesk,2025-01-03 23:52:00+05:30,email,A3002
20,TK-240021,legacy_fd,2025-01-03 23:52:00+05:30,email,A3002
23,TK-240025,helpdesk,2025-01-04 08:09:00+05:30,chat,A3010
24,TK-240025,legacy_fd,2025-01-04 08:09:00+05:30,chat,A3010
30,TK-240030,legacy_fd,2025-01-05 15:56:00+05:30,social,A3027
29,TK-240030,helpdesk,2025-01-05 15:56:00+05:30,social,A3027


In [ ]:
duplicate_tickets.groupby("ticket_id").agg(
    source_count=("source_system", "nunique"),
    created_count=("created_at", "nunique"),
    response_count=("first_response_at", "nunique"),
    agent_count=("agent_id", "nunique"),
    channel_count=("channel", "nunique")
).value_counts()

,,,,,count
source_count,created_count,response_count,agent_count,channel_count,
2,1,1,1,1,616


In [ ]:
tickets.duplicated(
    subset=[col for col in tickets.columns if col not in ["source_system"]],
    keep=False
).sum()

np.int64(560)

In [ ]:
duplicate_tickets = tickets[
    tickets["ticket_id"].duplicated(keep=False)
]

compare_cols = [
    col for col in tickets.columns
    if col not in ["ticket_id", "source_system"]
]

differences = duplicate_tickets.groupby("ticket_id")[
    compare_cols
].nunique(dropna=False)

differences.gt(1).sum().sort_values(ascending=False)

,0
csat_score,336
created_at,0
resolved_at,0
status,0
channel,0
first_response_at,0
customer_id,0
order_id,0
category,0
product_sku,0


In [ ]:
csat_differences = duplicate_tickets[
    duplicate_tickets["ticket_id"].isin(
        differences.index[differences["csat_score"] > 1]
    )
]

csat_differences[
    ["ticket_id", "source_system", "csat_score"]
].sort_values("ticket_id").head(20)

,ticket_id,source_system,csat_score
12,TK-240016,helpdesk,NaN
13,TK-240016,legacy_fd,0.0
15,TK-240018,helpdesk,NaN
16,TK-240018,legacy_fd,0.0
19,TK-240021,helpdesk,NaN
20,TK-240021,legacy_fd,0.0
23,TK-240025,helpdesk,NaN
24,TK-240025,legacy_fd,0.0
29,TK-240030,helpdesk,NaN
30,TK-240030,legacy_fd,0.0


In [ ]:
csat_differences.groupby(
    ["source_system", "csat_score"],
    dropna=False
).size()

,,0
source_system,csat_score,
helpdesk,NaN,336
legacy_fd,0.0,336


In [ ]:
# Legacy zero scores represent missing customer feedback
legacy_mask = (
    (tickets["source_system"] == "legacy_fd") &
    (tickets["csat_score"] == 0)
)

tickets.loc[legacy_mask, "csat_score"] = float("nan")

print("Legacy zero CSAT values normalized:", legacy_mask.sum())

Legacy zero CSAT values normalized: 2066


In [ ]:
duplicate_counts = tickets[
    tickets["ticket_id"].duplicated(keep=False)
].groupby("ticket_id").size()

print("Repeated ticket IDs:", len(duplicate_counts))
print("IDs appearing exactly twice:", (duplicate_counts == 2).sum())
print("IDs appearing more than twice:", (duplicate_counts > 2).sum())

Repeated ticket IDs: 616
IDs appearing exactly twice: 616
IDs appearing more than twice: 0


In [ ]:
# Prefer helpdesk records when the same ticket exists in both systems
tickets = tickets.sort_values(
    "source_system",
    key=lambda col: col.map({"legacy_fd": 0, "helpdesk": 1})
)

tickets = tickets.drop_duplicates(
    subset="ticket_id",
    keep="last"
).copy()

print("Rows after deduplication:", len(tickets))
print("Remaining duplicate ticket IDs:", tickets["ticket_id"].duplicated().sum())

Rows after deduplication: 11200
Remaining duplicate ticket IDs: 0


In [ ]:
tickets["customer_message"].value_counts().head(15)

,count
customer_message,
my order has not been delivered yet,10
tracking has said out for delivery for a week now,9
not pairing with my phone,9
still waiting for my refund,8
pickup scheduled but no one showed up,8
nobody came for the pickup,8
no update on my repair,7
since the last update the app just shows a white screen,7
received damaged product,7


In [ ]:
tickets["channel"].value_counts()

,count
channel,
chat,4959
email,3647
voice,1488
social,1106


In [ ]:
tickets[
    tickets["customer_message"].str.contains(
        "IVR|transcript|press 1|press 2|call failed|automated",
        case=False,
        na=False
    )
][["ticket_id", "channel", "customer_message", "agent_notes"]].head(25)

,ticket_id,channel,customer_message,agent_notes
9,TK-240012,voice,"[IVR transcript] pairing is not working, pleas...",Cx says pairing failure. Walked through forget...
5,TK-240008,voice,"[IVR transcript] ordered the wrong colour, don...",re: order cancellation | cancelled before disp...
84,TK-240082,voice,[IVR transcript] hello is the neckband compati...,issue: compatibility query | shared spec sheet...
77,TK-240074,voice,[IVR transcript] helo randoom disconnnects dur...,re: bt dropouts | advised to test in different...
97,TK-240095,voice,[IVR transcript] it's showing a spinning circl...,cx: update hang | walked through recovery mode...
95,TK-240092,voice,[IVR transcript] Honestly regretting this purc...,1. Rapid battery drain\n2. chk FW 3.4.4\n3. Es...
110,TK-240107,voice,[IVR transcript] HEY THE PARCEL LOOKED LIKE IT...,"1. transit damage\n2. photos received, damage ..."
129,TK-240129,voice,[IVR transcript] app crashes when I open devic...,re: app crashing | asked for phone model + os ...
183,TK-240179,voice,[IVR transcript] hii double payment deducted p...,cx: double charge | conf utrs -> duplicate ref...
176,TK-240171,voice,"[IVR transcript] Namaste, Ordered Pulse2 a mon...",Replacement raised after test. Issue: rapid ba...


In [ ]:
ivr_mask = tickets["customer_message"].str.startswith(
    "[IVR transcript]", na=False
)

print("IVR-labelled tickets:", ivr_mask.sum())

print("\nChannels:")
print(tickets.loc[ivr_mask, "channel"].value_counts())

print("\nStatuses:")
print(tickets.loc[ivr_mask, "status"].value_counts())

IVR-labelled tickets: 916

Channels:
channel
voice     902
chat        7
email       6
social      1
Name: count, dtype: int64

Statuses:
status
resolved    767
closed       83
open         42
pending      24
Name: count, dtype: int64


In [ ]:
pd.set_option("display.max_colwidth", 200)

tickets.loc[
    ivr_mask,
    ["ticket_id", "customer_message", "agent_notes"]
].sample(20, random_state=42)

,ticket_id,customer_message,agent_notes
5074,TK-245613,"[IVR transcript] paid on 08/10, still waiting for something to shhow up",re: shipment not received | confirmed address with cx -> refund initiated as lost in transit
11394,TK-253569,"[IVR transcript] dear team, i bought the strata 3 on may 11. the sound stutters whenever my phone is in my pocket. i reset both devices. nothing changed. i want my money back. thank you,",re: bt dropouts | advised to test in different location -> fw update resolved //KUN
4748,TK-245224,[IVR transcript] helo the return was accepetd but the amount is nowhere in my account pulse 2 buds VR882797 anyone there,re: rfnd delay | chk reverse pickup qc status -> arn shared w/ cx
4770,TK-245248,"[IVR transcript] I am losing patience. Bought the Pulsse 2 earbuds aorund October 02 from Amazon. need to change delivery address. I already tried editing in app. I want a replacement or refund, n...","reported: address chagne request. updated address w/ crr. already dispatched, advised to redirect w/ crr. follow-up not required."
4854,TK-245354,[IVR transcript] helo my phone just doesn't see it in the list anymore,"issue: device not discoverable | walked through forget + re-pair -> advised fw update via app, resolved"
6411,TK-247315,"[IVR transcript] Hello Vireo, Bought my Pulse 2 around 21 Nov from vireo.in. Packed the box a week ago and it's still here. I rescheduled pickup twice. I want my money back. Rgds Vivaan Mehta",issue: reverse pkp pending | chk pkp awb -> refunded rs 6298
8865,TK-250421,"[IVR transcript] Hi there, I buoght my Pulse 2 on February 22. Package not delivered even ater 15 days. I checked the tracking page daily. Kindly look inot it. Please revert Karan Pawar","cx says order not delivered. confirmed address with cx. rto confirmed, reshipped."
7806,TK-249061,"[IVR transcript] Hi team, Ordered my Orbit 10 days ago. The Vireo app keeps crashing. Please resolve asap. Awaiting response Simran Yadav","re: app crash on device page | known issue on android 15 - bug logged -> advised beta build, resolved"
7219,TK-248334,[IVR transcript] sir ji the invoice pdf link gievs a 404 2 weeks ago se the fit band,1. Invoice not downloading\n2. Updated GSTIN on order\n3. Invoice emailed
507,TK-240552,[IVR transcript] battery life has dropped to almost nothing,issue: poor battery backup | checked fw 1.6.11 -> escalated to warranty


In [ ]:
failure_pattern = (
    r"failed ivr|ivr failed|call failed|"
    r"abandoned call|call abandoned|"
    r"no agent connected|agent not connected|"
    r"no human response|disconnected before agent"
)

failure_mask = (
    tickets["customer_message"].str.contains(
        failure_pattern, case=False, na=False
    )
    |
    tickets["agent_notes"].str.contains(
        failure_pattern, case=False, na=False
    )
)

print("Records matching failure indicators:", failure_mask.sum())

display(
    tickets.loc[
        failure_mask,
        ["ticket_id", "channel", "customer_message", "agent_notes"]
    ].head(20)
)

Records matching failure indicators: 0


,ticket_id,channel,customer_message,agent_notes


In [ ]:
failure_pattern = (
    r"fail|disconnect|abandon|timeout|"
    r"no response|no input|no speech|"
    r"hang.?up|unavailable|could not connect|"
    r"couldn't connect|call ended|transcript unavailable"
)

voice_tickets = tickets[tickets["channel"] == "voice"]

candidate_mask = (
    voice_tickets["customer_message"].str.contains(
        failure_pattern, case=False, na=False
    )
    |
    voice_tickets["agent_notes"].str.contains(
        failure_pattern, case=False, na=False
    )
)

print("Voice tickets:", len(voice_tickets))
print("Possible failure candidates:", candidate_mask.sum())

display(
    voice_tickets.loc[
        candidate_mask,
        ["ticket_id", "status", "customer_message", "agent_notes"]
    ].head(30)
)

Voice tickets: 1488
Possible failure candidates: 116


,ticket_id,status,customer_message,agent_notes
9,TK-240012,resolved,"[IVR transcript] pairing is not working, please help","Cx says pairing failure. Walked through forget + re-pair. Advised FW update via app, resolved. Cx satisfied on chat. -RK"
64,TK-240059,resolved,"update failed and now it won't turn on, pls resollve asap",cx: fw update stuck | asked cx to keep in case 30 min -> replacement raised (bricked)
4277,TK-244644,resolved,random disconnects during calls,"re: connection dropping | checked interference sources -> advised, cx will monitor"
4275,TK-244642,closed,[IVR transcript] i am losing patience. order vr896377 (orbit mini). bluetooth pairing fails every time. i already tried on another phone. refund. now,"Cx reported pairing failure. Asked cx to reset the device. Advised FW update via app, rslvd. Cx satisfied on chat."
452,TK-240489,resolved,"team,\nmoney debited but order not showing\nbahtu pareshan hu\nmy strata 2 headphones vr899488",Order manually created. Issue: failed order after payment. Checked PG dashboard for txn. [closed]
608,TK-240664,closed,the pairing light blinks but nothing ever shows up on my iphone\nvr881778,"re: pairing failure | walked through forget + re-pair -> reset done, paired ok"
725,TK-240782,resolved,This is very disappointing. Bought the mini speaker around February 21 from Flipkart. the page failed after I paid and now nothing shows in my account. I already tried logging out and in. Escalate...,"Full refund processed (375). Issue: payment debited, no order. Checked PG dashboard for txn. [closed]"
769,TK-240831,resolved,[IVR transcript] random disconnects durng calls pleaase call me on my registered number,1. Intermittent disconnects\n2. chk FW 2.3.7\n3. Replacement raised
906,TK-240981,resolved,"[IVR transcript] Hi, Got Strata 3 from Amazon in February. Money debited but ordder not showing. I waited 24 hours. How do I get this fixed?",cx: failed ord after payment | chk pg dashboard for txn -> refunded rs 6999
1027,TK-241111,resolved,Pathetic experience honestly. This is regarding the mini speaker. bluetooth pairing fails every time. I already forgot the device and tried again. Refund. Now,"Cx reported device not discoverable. Walked through forget + re-pair. adv FW update via app, rslvd. Case closed with cx consent. -RK"


In [ ]:
agents = pd.read_csv("agents.csv")

print("Shape:", agents.shape)
print("\nColumns:")
print(agents.columns.tolist())

display(agents.head(10))

Shape: (46, 8)

Columns:
['agent_id', 'name', 'site', 'team', 'shift', 'tier', 'from_date', 'to_date']


,agent_id,name,site,team,shift,tier,from_date,to_date
0,A3001,Zoya Mehta,Indore,Chat Frontline,Night,1,2021-04-20,2025-06-29
1,A3002,Tarun Mishra,Indore,Chat Frontline,Night,1,2023-02-12,2025-06-29
2,A3002,Tarun Mishra,Indore,Chat Frontline,Day,1,2025-06-30,NaN
3,A3003,Harpreet Deshpande,Indore,Chat Frontline,Night,1,2023-06-14,2025-06-29
4,A3003,Harpreet Deshpande,Indore,Chat Frontline,Day,1,2025-06-30,NaN
5,A3004,Krishna Naidu,Bengaluru,Chat Frontline,Day,1,2022-12-30,NaN
6,A3005,Sameer Joshi,Bengaluru,Chat Frontline,Morning,1,2021-07-28,NaN
7,A3006,Lakshmi Reddy,Bengaluru,Chat Frontline,Day,1,2021-02-05,NaN
8,A3007,Neha Gaikwad,Bengaluru,Chat Frontline,Morning,1,2023-09-10,NaN
9,A3008,Steven Kaur,Indore,Chat Frontline,Morning,1,2022-05-02,NaN


In [ ]:
agents["from_date"] = pd.to_datetime(agents["from_date"])

agents["to_date"] = pd.to_datetime(
    agents["to_date"],
    errors="coerce"
)

print("Shift counts:")
print(agents["shift"].value_counts())

print("\nMissing end dates:", agents["to_date"].isna().sum())

print("\nAgents with multiple roster rows:")
print(agents["agent_id"].value_counts().loc[lambda x: x > 1])

Shift counts:
shift
Day        21
Morning    20
Night       5
Name: count, dtype: int64

Missing end dates: 41

Agents with multiple roster rows:
agent_id
A3002    2
A3003    2
Name: count, dtype: int64


In [ ]:
roster_check = agents.sort_values(
    ["agent_id", "from_date"]
).copy()

roster_check["previous_to_date"] = (
    roster_check.groupby("agent_id")["to_date"].shift()
)

roster_check["next_from_date"] = (
    roster_check.groupby("agent_id")["from_date"].shift(-1)
)

display(
    roster_check.loc[
        roster_check["next_from_date"].notna(),
        ["agent_id", "shift", "from_date", "to_date", "next_from_date"]
    ]
)

,agent_id,shift,from_date,to_date,next_from_date
1,A3002,Night,2023-02-12,2025-06-29,2025-06-30
3,A3003,Night,2023-06-14,2025-06-29,2025-06-30


In [ ]:
# Check whether ticket agents exist in the roster
roster_agent_ids = set(agents["agent_id"].dropna())
ticket_agent_ids = set(tickets["agent_id"].dropna())

print("Unique ticket agents:", len(ticket_agent_ids))
print("Unique roster agents:", len(roster_agent_ids))
print("Ticket agents missing from roster:",
      sorted(ticket_agent_ids - roster_agent_ids))

# Check tickets without an agent ID
print("\nTickets missing agent_id:", tickets["agent_id"].isna().sum())

# Check tickets without a resolution timestamp
print("Tickets missing resolved_at:", tickets["resolved_at"].isna().sum())

# Check whether ticket dates fall within the dataset's period
print("\nTicket creation range:")
print(tickets["created_at"].min(), "to", tickets["created_at"].max())

print("\nTicket resolution range:")
print(tickets["resolved_at"].min(), "to", tickets["resolved_at"].max())

Unique ticket agents: 44
Unique roster agents: 44
Ticket agents missing from roster: []

Tickets missing agent_id: 0
Tickets missing resolved_at: 589

Ticket creation range:
2025-01-01 11:42:00+05:30 to 2026-06-30 21:19:00+05:30

Ticket resolution range:
2025-01-01 15:15:00+05:30 to 2026-07-07 17:30:00+05:30


In [ ]:
# Create a working copy
roster = agents.copy()
ticket_work = tickets.copy()

# Ensure dates are parsed correctly
roster["from_date"] = pd.to_datetime(roster["from_date"]).dt.date
roster["to_date"] = pd.to_datetime(roster["to_date"], errors="coerce").dt.date

ticket_work["resolution_date"] = (
    ticket_work["resolved_at"].dt.date
)

# Match tickets to all roster rows for the same agent
mapped = ticket_work.merge(
    roster[["agent_id", "name", "team", "shift",
            "from_date", "to_date"]],
    on="agent_id",
    how="left",
    suffixes=("", "_roster")
)

# Keep the roster assignment active on the resolution date
active = (
    mapped["resolution_date"].notna()
    & (mapped["resolution_date"] >= mapped["from_date"])
    & (
        mapped["to_date"].isna()
        | (mapped["resolution_date"] <= mapped["to_date"])
    )
)

mapped = mapped.loc[active].copy()

# Check matching results
print("Tickets matched to a historical roster assignment:", len(mapped))
print("Unique ticket IDs matched:", mapped["ticket_id"].nunique())

print("\nTickets without a resolution date:",
      ticket_work["resolved_at"].isna().sum())

print("\nTickets with a resolution date but no roster match:",
      ticket_work.loc[
          ticket_work["resolved_at"].notna(), "ticket_id"
      ].nunique() - mapped["ticket_id"].nunique())

print("\nShift distribution among matched tickets:")
print(mapped["shift"].value_counts(dropna=False))

Tickets matched to a historical roster assignment: 10611
Unique ticket IDs matched: 10611

Tickets without a resolution date: 589

Tickets with a resolution date but no roster match: 0

Shift distribution among matched tickets:
shift
Morning    5928
Day        4428
Night       255
Name: count, dtype: int64


In [ ]:
# Check first-response data quality
print("Missing first_response_at:",
      tickets["first_response_at"].isna().sum())

print("Missing response duration:",
      tickets["response_minutes"].isna().sum())

print("Negative response durations:",
      (tickets["response_minutes"] < 0).sum())

print("\nChannel counts:")
print(tickets["channel"].value_counts(dropna=False))

print("\nChannels without an SLA target:")
print(
    tickets.loc[
        tickets["sla_minutes"].isna(), "channel"
    ].value_counts(dropna=False)
)

print("\nResponse duration summary (minutes):")
print(tickets["response_minutes"].describe())

print("\nPreliminary SLA results:")
print(tickets["sla_breach"].value_counts(dropna=False))

print("\nPreliminary breach counts by channel:")
print(
    tickets.groupby("channel", dropna=False)["sla_breach"]
    .agg(["count", "sum"])
)

Missing first_response_at: 0
Missing response duration: 0
Negative response durations: 0

Channel counts:
channel
chat      4959
email     3647
voice     1488
social    1106
Name: count, dtype: int64

Channels without an SLA target:
Series([], Name: count, dtype: int64)

Response duration summary (minutes):
count    11200.000000
mean       155.174643
std        229.881160
min          0.000000
25%          7.000000
50%         51.000000
75%        241.000000
max       5155.000000
Name: response_minutes, dtype: float64

Preliminary SLA results:
sla_breach
False    8760
True     2440
Name: count, dtype: int64

Preliminary breach counts by channel:
         count   sum
channel             
chat      4959  1368
email     3647   745
social    1106   243
voice     1488    84


In [ ]:
# Search voice tickets for possible failed-call indicators
voice = tickets[tickets["channel"].eq("voice")].copy()

failure_pattern = (
    r"fail|disconnect|abandon|timeout|no response|no input|"
    r"no speech|hang.?up|unavailable|could not connect|"
    r"couldn't connect|call ended|transcript unavailable"
)

voice["possible_failure"] = (
    voice["customer_message"].str.contains(
        failure_pattern, case=False, na=False, regex=True
    )
    | voice["agent_notes"].str.contains(
        failure_pattern, case=False, na=False, regex=True
    )
)

print("Total voice tickets:", len(voice))
print("Possible failure indicators:", voice["possible_failure"].sum())

display(
    voice.loc[
        voice["possible_failure"],
        [
            "ticket_id",
            "created_at",
            "first_response_at",
            "resolved_at",
            "status",
            "customer_message",
            "agent_notes",
            "source_system"
        ]
    ].head(25)
)

Total voice tickets: 1488
Possible failure indicators: 116


,ticket_id,created_at,first_response_at,resolved_at,status,customer_message,agent_notes,source_system
9,TK-240012,2025-01-02 19:45:00+05:30,2025-01-02 19:59:00+05:30,2025-01-02 20:21:00+05:30,resolved,"[IVR transcript] pairing is not working, please help","Cx says pairing failure. Walked through forget + re-pair. Advised FW update via app, resolved. Cx satisfied on chat. -RK",legacy_fd
64,TK-240059,2025-01-09 18:41:00+05:30,2025-01-09 18:58:00+05:30,2025-01-09 19:17:00+05:30,resolved,"update failed and now it won't turn on, pls resollve asap",cx: fw update stuck | asked cx to keep in case 30 min -> replacement raised (bricked),legacy_fd
4277,TK-244644,2025-09-11 19:52:00+05:30,2025-09-11 20:34:00+05:30,2025-09-11 20:49:00+05:30,resolved,random disconnects during calls,"re: connection dropping | checked interference sources -> advised, cx will monitor",legacy_fd
4275,TK-244642,2025-09-11 16:54:00+05:30,2025-09-11 17:18:00+05:30,2025-09-11 17:46:00+05:30,closed,[IVR transcript] i am losing patience. order vr896377 (orbit mini). bluetooth pairing fails every time. i already tried on another phone. refund. now,"Cx reported pairing failure. Asked cx to reset the device. Advised FW update via app, rslvd. Cx satisfied on chat.",legacy_fd
452,TK-240489,2025-02-10 15:41:00+05:30,2025-02-10 15:54:00+05:30,2025-02-10 16:32:00+05:30,resolved,"team,\nmoney debited but order not showing\nbahtu pareshan hu\nmy strata 2 headphones vr899488",Order manually created. Issue: failed order after payment. Checked PG dashboard for txn. [closed],legacy_fd
608,TK-240664,2025-02-21 16:36:00+05:30,2025-02-21 16:44:00+05:30,2025-02-21 17:20:00+05:30,closed,the pairing light blinks but nothing ever shows up on my iphone\nvr881778,"re: pairing failure | walked through forget + re-pair -> reset done, paired ok",legacy_fd
725,TK-240782,2025-02-28 17:09:00+05:30,2025-02-28 17:17:00+05:30,2025-02-28 17:39:00+05:30,resolved,This is very disappointing. Bought the mini speaker around February 21 from Flipkart. the page failed after I paid and now nothing shows in my account. I already tried logging out and in. Escalate...,"Full refund processed (375). Issue: payment debited, no order. Checked PG dashboard for txn. [closed]",legacy_fd
769,TK-240831,2025-03-03 11:07:00+05:30,2025-03-03 11:17:00+05:30,2025-03-03 11:53:00+05:30,resolved,[IVR transcript] random disconnects durng calls pleaase call me on my registered number,1. Intermittent disconnects\n2. chk FW 2.3.7\n3. Replacement raised,legacy_fd
906,TK-240981,2025-03-12 11:50:00+05:30,2025-03-12 12:08:00+05:30,2025-03-12 12:42:00+05:30,resolved,"[IVR transcript] Hi, Got Strata 3 from Amazon in February. Money debited but ordder not showing. I waited 24 hours. How do I get this fixed?",cx: failed ord after payment | chk pg dashboard for txn -> refunded rs 6999,legacy_fd
1027,TK-241111,2025-03-20 17:38:00+05:30,2025-03-20 18:17:00+05:30,2025-03-20 18:38:00+05:30,resolved,Pathetic experience honestly. This is regarding the mini speaker. bluetooth pairing fails every time. I already forgot the device and tried again. Refund. Now,"Cx reported device not discoverable. Walked through forget + re-pair. adv FW update via app, rslvd. Case closed with cx consent. -RK",legacy_fd


In [ ]:
# Search specifically for IVR/call handling failures
ivr_failure_pattern = (
    r"ivr.{0,30}(fail|disconnect|abandon|timeout|"
    r"no input|no speech|unavailable|not connect)"
    r"|"
    r"(fail|disconnect|abandon|timeout|unavailable)"
    r".{0,30}ivr"
    r"|"
    r"call (was )?(abandoned|disconnected|dropped|"
    r"failed|not connected)"
    r"|"
    r"no agent connected|agent never connected"
    r"|"
    r"transcript unavailable|empty transcript"
)

voice["ivr_failure_candidate"] = (
    voice["customer_message"].str.contains(
        ivr_failure_pattern, case=False, na=False, regex=True
    )
    | voice["agent_notes"].str.contains(
        ivr_failure_pattern, case=False, na=False, regex=True
    )
)

candidates = voice.loc[
    voice["ivr_failure_candidate"],
    [
        "ticket_id",
        "created_at",
        "first_response_at",
        "status",
        "customer_message",
        "agent_notes",
        "source_system"
    ]
]

print("Narrow IVR/call failure candidates:", len(candidates))
display(candidates.head(30))

Narrow IVR/call failure candidates: 14


/tmp/ipykernel_3448/646860408.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  voice["customer_message"].str.contains(
/tmp/ipykernel_3448/646860408.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  | voice["agent_notes"].str.contains(


,ticket_id,created_at,first_response_at,status,customer_message,agent_notes,source_system
769,TK-240831,2025-03-03 11:07:00+05:30,2025-03-03 11:17:00+05:30,resolved,[IVR transcript] random disconnects durng calls pleaase call me on my registered number,1. Intermittent disconnects\n2. chk FW 2.3.7\n3. Replacement raised,legacy_fd
2190,TK-242353,2025-06-02 09:40:00+05:30,2025-06-02 10:51:00+05:30,resolved,[IVR transcript] random disconnects during clals please help,1. Intermittent disconnects\n2. chk FW 1.2.10\n3. Replacement raised,legacy_fd
2250,TK-242417,2025-06-06 19:57:00+05:30,2025-06-06 20:52:00+05:30,open,"[IVR transcript] cannot connect the speaker to wifi, need this sorted this week",adv router settings. Issue: wifi setup failing. Asked cx to try 2.4GHz only. -RK,legacy_fd
12,TK-240016,2025-01-03 09:08:00+05:30,2025-01-03 09:27:00+05:30,resolved,[IVR transcript] helo teh page failed after i paid and now nothing shows in my account pls reply,cx: failed order after payment | asked for utr -> order manually created,helpdesk
7515,TK-248698,2026-01-08 19:51:00+05:30,2026-01-08 20:26:00+05:30,resolved,[IVR transcript] audio keeps disconnecting every few minutes,"Cx says BT dropouts. Checked FW 3.6.6. Advised, cx will monitor.",helpdesk
8202,TK-249569,2026-02-03 18:34:00+05:30,2026-02-03 19:39:00+05:30,resolved,[IVR transcript] random disconnects during calls vr908727,Contact re connection dropping. adv to test in different location. rplc raised. Will monitor. -RK,helpdesk
8860,TK-250414,2026-03-03 12:42:00+05:30,2026-03-03 14:58:00+05:30,resolved,[IVR transcript] hi sir the page failed after i paid and now nothing shows in my account my nexa fit band hello??,closed [closed],helpdesk
9394,TK-251082,2026-03-23 20:36:00+05:30,2026-03-23 21:19:00+05:30,resolved,[IVR transcript] hii the page failed after i paid and now nothing shows in my account the charging case i ordered vr891398 hello??,1. Failed order after payment\n2. Asked for UTR\n3. Refund processed to source,helpdesk
9910,TK-251718,2026-04-14 18:55:00+05:30,2026-04-14 20:25:00+05:30,resolved,[IVR transcript] the page failed after I padi and now nothing shows in my account. please advise,auto-refund confirmed within 5-7 days -RK,helpdesk
10779,TK-252800,2026-05-20 10:26:00+05:30,2026-05-20 10:52:00+05:30,resolved,[IVR transcript] the page failed after I paid and now nothing shows in my account,"1. Failed ord after payment\n2. conf w/ payment gateway\n3. rfnd approved, Rs 840, 5-7 working days",helpdesk


In [ ]:
# Start with tickets successfully matched to a historical roster assignment
report = mapped.copy()

# Define the reporting week: Monday in IST
report["week_start"] = (
    report["created_at"].dt.normalize()
    - pd.to_timedelta(report["created_at"].dt.dayofweek, unit="D")
).dt.strftime("%Y-%m-%d")

# Summarise weekly SLA performance by resolving agent and shift
weekly_report = (
    report.groupby(
        ["week_start", "agent_id", "name", "shift"],
        dropna=False
    )
    .agg(
        tickets=("ticket_id", "nunique"),
        breaches=("sla_breach", "sum"),
        avg_response_minutes=("response_minutes", "mean")
    )
    .reset_index()
)

weekly_report["breach_rate_pct"] = (
    100 * weekly_report["breaches"] / weekly_report["tickets"]
).round(2)

weekly_report["estimated_credit_inr"] = (
    weekly_report["breaches"] * 350
).astype(int)

# Sort chronologically, then by agent
weekly_report = weekly_report.sort_values(
    ["week_start", "agent_id"]
).reset_index(drop=True)

print("Weekly report rows:", len(weekly_report))
print("Weeks covered:", weekly_report["week_start"].nunique())
print("Tickets included:", weekly_report["tickets"].sum())
print("Breaches included:", weekly_report["breaches"].sum())
print(
    "Estimated credits (INR):",
    weekly_report["estimated_credit_inr"].sum()
)

display(weekly_report.head(15))

Weekly report rows: 2903
Weeks covered: 79
Tickets included: 10611
Breaches included: 2320
Estimated credits (INR): 812000


,week_start,agent_id,name,shift,tickets,breaches,avg_response_minutes,breach_rate_pct,estimated_credit_inr
0,2024-12-30,A3002,Tarun Mishra,Night,2,0,184.5,0.0,0
1,2024-12-30,A3006,Lakshmi Reddy,Day,2,0,24.0,0.0,0
2,2024-12-30,A3010,Neha Chatterjee,Morning,1,0,2.0,0.0,0
3,2024-12-30,A3015,Yash Agarwal,Day,1,0,33.0,0.0,0
4,2024-12-30,A3016,Deepak Jadhav,Night,1,0,4.0,0.0,0
5,2024-12-30,A3017,Shreya Bhatt,Night,1,0,17.0,0.0,0
6,2024-12-30,A3022,Steven Tiwari,Day,1,0,144.0,0.0,0
7,2024-12-30,A3023,Rahul Khanna,Day,2,0,12.5,0.0,0
8,2024-12-30,A3024,Kunal Khanna,Morning,2,0,14.5,0.0,0
9,2024-12-30,A3027,Dev Chatterjee,Day,1,0,95.0,0.0,0


In [ ]:
# Validate that the weekly report reconciles with its source data

print("TICKET RECONCILIATION")
print("Tickets in mapped data:", mapped["ticket_id"].nunique())
print("Tickets in weekly report:", weekly_report["tickets"].sum())

print("\nSLA BREACH RECONCILIATION")
print("Breaches in mapped data:", int(mapped["sla_breach"].sum()))
print("Breaches in weekly report:", int(weekly_report["breaches"].sum()))

print("\nCREDIT RECONCILIATION")
print(
    "Expected estimated credits (INR):",
    int(mapped["sla_breach"].sum()) * 350
)
print(
    "Weekly report estimated credits (INR):",
    int(weekly_report["estimated_credit_inr"].sum())
)

print("\nCHECKS")
print(
    "Ticket totals match:",
    mapped["ticket_id"].nunique() == weekly_report["tickets"].sum()
)
print(
    "Breach totals match:",
    int(mapped["sla_breach"].sum()) == int(weekly_report["breaches"].sum())
)
print(
    "Credit totals match:",
    int(mapped["sla_breach"].sum()) * 350
    == int(weekly_report["estimated_credit_inr"].sum())
)

TICKET RECONCILIATION
Tickets in mapped data: 10611
Tickets in weekly report: 10611

SLA BREACH RECONCILIATION
Breaches in mapped data: 2320
Breaches in weekly report: 2320

CREDIT RECONCILIATION
Expected estimated credits (INR): 812000
Weekly report estimated credits (INR): 812000

CHECKS
Ticket totals match: True
Breach totals match: True
Credit totals match: True


In [ ]:
# 1. Overall weekly SLA trend
weekly_trend = (
    weekly_report.groupby("week_start")
    .agg(
        tickets=("tickets", "sum"),
        breaches=("breaches", "sum"),
        estimated_credits_inr=("estimated_credit_inr", "sum")
    )
    .reset_index()
)

weekly_trend["breach_rate_pct"] = (
    100 * weekly_trend["breaches"] / weekly_trend["tickets"]
).round(2)

print("LATEST 12 WEEKS")
display(weekly_trend.tail(12))

# 2. Aggregate by agent and shift across the full reporting period
agent_shift_summary = (
    weekly_report.groupby(
        ["agent_id", "name", "shift"],
        dropna=False
    )
    .agg(
        tickets=("tickets", "sum"),
        breaches=("breaches", "sum"),
        estimated_credits_inr=("estimated_credit_inr", "sum")
    )
    .reset_index()
)

agent_shift_summary["breach_rate_pct"] = (
    100 * agent_shift_summary["breaches"]
    / agent_shift_summary["tickets"]
).round(2)

# Filter out very small samples for the priority review
priority_review = (
    agent_shift_summary[
        agent_shift_summary["tickets"] >= 30
    ]
    .sort_values(
        ["breaches", "breach_rate_pct"],
        ascending=False
    )
)

print("\nTOP 10 AGENT-SHIFT GROUPS BY BREACH COUNT")
display(priority_review.head(10))

# 3. Overall totals
total_tickets = weekly_trend["tickets"].sum()
total_breaches = weekly_trend["breaches"].sum()

print("\nOVERALL SUMMARY")
print("Tickets:", total_tickets)
print("Breaches:", total_breaches)
print("Breach rate (%):", round(100 * total_breaches / total_tickets, 2))

LATEST 12 WEEKS


,week_start,tickets,breaches,estimated_credits_inr,breach_rate_pct
67,2026-04-13,164,41,14350,25.00
68,2026-04-20,163,48,16800,29.45
69,2026-04-27,173,42,14700,24.28
70,2026-05-04,145,33,11550,22.76
71,2026-05-11,161,44,15400,27.33
72,2026-05-18,169,44,15400,26.04
73,2026-05-25,146,38,13300,26.03
74,2026-06-01,152,34,11900,22.37
75,2026-06-08,154,46,16100,29.87
76,2026-06-15,186,46,16100,24.73



TOP 10 AGENT-SHIFT GROUPS BY BREACH COUNT


,agent_id,name,shift,tickets,breaches,estimated_credits_inr,breach_rate_pct
38,A3037,Rahul Sen,Morning,569,172,60200,30.23
29,A3028,Krishna Kaur,Morning,530,167,58450,31.51
32,A3031,Om Malhotra,Morning,541,155,54250,28.65
19,A3018,Farah Sheikh,Morning,505,146,51100,28.91
20,A3019,Sai Sandhu,Morning,493,135,47250,27.38
33,A3032,Aadhya Sheikh,Morning,348,130,45500,37.36
36,A3035,Rohit Mukherjee,Morning,324,105,36750,32.41
34,A3033,Amit Kulkarni,Morning,307,102,35700,33.22
14,A3013,Ritika Chatterjee,Morning,226,94,32900,41.59
11,A3010,Neha Chatterjee,Morning,208,87,30450,41.83



OVERALL SUMMARY
Tickets: 10611
Breaches: 2320
Breach rate (%): 21.86


In [ ]:
# Channel-level SLA performance across all deduplicated tickets
channel_summary = (
    tickets.groupby("channel")
    .agg(
        total_tickets=("ticket_id", "nunique"),
        breaches=("sla_breach", "sum"),
        avg_response_minutes=("response_minutes", "mean")
    )
    .reset_index()
)

channel_summary["breach_rate_pct"] = (
    100 * channel_summary["breaches"]
    / channel_summary["total_tickets"]
).round(2)

channel_summary["estimated_credit_inr"] = (
    channel_summary["breaches"] * 350
).astype(int)

channel_summary = channel_summary.sort_values(
    "breach_rate_pct", ascending=False
)

display(channel_summary)

# Check how much of the full dataset is covered by the roster-mapped report
print("\nPOPULATION COVERAGE")
print("All deduplicated tickets:", tickets["ticket_id"].nunique())
print("Tickets with a resolution timestamp:",
      tickets["resolved_at"].notna().sum())
print("Tickets mapped to historical roster:", mapped["ticket_id"].nunique())
print("Tickets without a resolution timestamp:",
      tickets["resolved_at"].isna().sum())

print("\nUNRESOLVED TICKETS BY STATUS")
display(
    tickets.loc[tickets["resolved_at"].isna(), "status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="ticket_count")
)

,channel,total_tickets,breaches,avg_response_minutes,breach_rate_pct,estimated_credit_inr
0,chat,4959,1368,85.151240,27.59,478800
2,social,1106,243,152.216998,21.97,85050
1,email,3647,745,295.752399,20.43,260750
3,voice,1488,84,46.189516,5.65,29400



POPULATION COVERAGE
All deduplicated tickets: 11200
Tickets with a resolution timestamp: 10611
Tickets mapped to historical roster: 10611
Tickets without a resolution timestamp: 589

UNRESOLVED TICKETS BY STATUS


,status,ticket_count
0,open,353
1,pending,236


In [ ]:
# Use the resolved tickets already matched to historical roster assignments
impact = mapped.copy()

# Assign quarters based on ticket creation date in IST
impact["quarter"] = (
    impact["created_at"]
    .dt.tz_localize(None)
    .dt.to_period("Q")
    .astype(str)
)

quarterly_impact = (
    impact.groupby("quarter")
    .agg(
        resolved_tickets=("ticket_id", "nunique"),
        sla_breaches=("sla_breach", "sum")
    )
    .reset_index()
)

quarterly_impact["breach_rate_pct"] = (
    100 * quarterly_impact["sla_breaches"]
    / quarterly_impact["resolved_tickets"]
).round(2)

quarterly_impact["estimated_credit_inr"] = (
    quarterly_impact["sla_breaches"] * 350
).astype(int)

# Illustrative goal: reduce SLA breaches by 20%
quarterly_impact["illustrative_20pct_reduction"] = (
    quarterly_impact["sla_breaches"] * 0.20
).round().astype(int)

quarterly_impact["potential_credit_avoidance_inr"] = (
    quarterly_impact["illustrative_20pct_reduction"] * 350
).astype(int)

display(quarterly_impact)

print("\nIMPORTANT")
print("The 20% reduction is a proposed target, not a measured result.")
print("Potential avoidance assumes each prevented breach avoids a ₹350 credit.")
print("Actual savings require validation against Finance's credit records.")

,quarter,resolved_tickets,sla_breaches,breach_rate_pct,estimated_credit_inr,illustrative_20pct_reduction,potential_credit_avoidance_inr
0,2025Q1,972,93,9.57,32550,19,6650
1,2025Q2,1172,104,8.87,36400,21,7350
2,2025Q3,1744,454,26.03,158900,91,31850
3,2025Q4,2444,592,24.22,207200,118,41300
4,2026Q1,2179,543,24.92,190050,109,38150
5,2026Q2,2100,534,25.43,186900,107,37450



IMPORTANT
The 20% reduction is a proposed target, not a measured result.
Potential avoidance assumes each prevented breach avoids a ₹350 credit.
Actual savings require validation against Finance's credit records.


In [ ]:
import pandas as pd

# 1. Weekly report: Monday-start weeks, based on ticket creation in IST
report = mapped.copy()

report["week_start"] = (
    report["created_at"].dt.normalize()
    - pd.to_timedelta(report["created_at"].dt.dayofweek, unit="D")
).dt.strftime("%Y-%m-%d")

weekly_report = (
    report.groupby(
        ["week_start", "agent_id", "name", "shift"],
        dropna=False
    )
    .agg(
        tickets=("ticket_id", "nunique"),
        breaches=("sla_breach", "sum"),
        avg_response_minutes=("response_minutes", "mean")
    )
    .reset_index()
)

weekly_report["breach_rate_pct"] = (
    100 * weekly_report["breaches"] / weekly_report["tickets"]
).round(2)

weekly_report["estimated_credit_inr"] = (
    weekly_report["breaches"] * 350
).astype(int)

weekly_report["avg_response_minutes"] = (
    weekly_report["avg_response_minutes"].round(2)
)

weekly_report = weekly_report.sort_values(
    ["week_start", "agent_id"]
)

# 2. Channel report: resolved-ticket population only
channel_report = (
    mapped.groupby("channel")
    .agg(
        resolved_tickets=("ticket_id", "nunique"),
        breaches=("sla_breach", "sum"),
        avg_response_minutes=("response_minutes", "mean")
    )
    .reset_index()
)

channel_report["breach_rate_pct"] = (
    100 * channel_report["breaches"]
    / channel_report["resolved_tickets"]
).round(2)

channel_report["estimated_credit_inr"] = (
    channel_report["breaches"] * 350
).astype(int)

channel_report["avg_response_minutes"] = (
    channel_report["avg_response_minutes"].round(2)
)

# 3. Export reports
weekly_report.to_csv("weekly_sla_report.csv", index=False)
channel_report.to_csv("channel_sla_report.csv", index=False)

# 4. Basic validation checks
assert weekly_report["tickets"].sum() == mapped["ticket_id"].nunique()
assert weekly_report["breaches"].sum() == int(mapped["sla_breach"].sum())
assert not weekly_report.duplicated(
    ["week_start", "agent_id", "name", "shift"]
).any()

print("Reports exported successfully.")
print("Weekly report rows:", len(weekly_report))
print("Channel report rows:", len(channel_report))
print("Resolved-ticket population:", mapped["ticket_id"].nunique())
print("SLA breaches in report:", int(mapped["sla_breach"].sum()))

display(weekly_report.head())
display(channel_report)

Reports exported successfully.
Weekly report rows: 2903
Channel report rows: 4
Resolved-ticket population: 10611
SLA breaches in report: 2320


,week_start,agent_id,name,shift,tickets,breaches,avg_response_minutes,breach_rate_pct,estimated_credit_inr
0,2024-12-30,A3002,Tarun Mishra,Night,2,0,184.5,0.0,0
1,2024-12-30,A3006,Lakshmi Reddy,Day,2,0,24.0,0.0,0
2,2024-12-30,A3010,Neha Chatterjee,Morning,1,0,2.0,0.0,0
3,2024-12-30,A3015,Yash Agarwal,Day,1,0,33.0,0.0,0
4,2024-12-30,A3016,Deepak Jadhav,Night,1,0,4.0,0.0,0


,channel,resolved_tickets,breaches,avg_response_minutes,breach_rate_pct,estimated_credit_inr
0,chat,4721,1305,85.50,27.64,456750
1,email,3462,709,295.03,20.48,248150
2,social,1032,225,150.93,21.80,78750
3,voice,1396,81,46.18,5.80,28350


In [ ]:
import pandas as pd

# Build a validation checklist from the processed data
validation_checks = [
    {
        "check": "Duplicate ticket IDs after deduplication",
        "result": int(tickets["ticket_id"].duplicated().sum()),
        "expected": "0",
        "passed": tickets["ticket_id"].is_unique
    },
    {
        "check": "Missing first-response timestamps",
        "result": int(tickets["first_response_at"].isna().sum()),
        "expected": "0",
        "passed": tickets["first_response_at"].notna().all()
    },
    {
        "check": "Negative response durations",
        "result": int((tickets["response_minutes"] < 0).sum()),
        "expected": "0",
        "passed": not (tickets["response_minutes"] < 0).any()
    },
    {
        "check": "Tickets missing SLA targets",
        "result": int(tickets["sla_minutes"].isna().sum()),
        "expected": "0",
        "passed": tickets["sla_minutes"].notna().all()
    },
    {
        "check": "Resolved tickets without historical roster match",
        "result": int(
            tickets.loc[tickets["resolved_at"].notna(), "agent_id"]
            .isin(agents["agent_id"]).eq(False).sum()
        ),
        "expected": "0",
        "passed": tickets.loc[
            tickets["resolved_at"].notna(), "agent_id"
        ].isin(agents["agent_id"]).all()
    },
    {
        "check": "Duplicate rows in weekly report grouping",
        "result": int(
            weekly_report.duplicated(
                ["week_start", "agent_id", "name", "shift"]
            ).sum()
        ),
        "expected": "0",
        "passed": not weekly_report.duplicated(
            ["week_start", "agent_id", "name", "shift"]
        ).any()
    }
]

validation_df = pd.DataFrame(validation_checks)
validation_df["status"] = validation_df["passed"].map(
    {True: "PASS", False: "REVIEW"}
)

display(validation_df)
validation_df.to_csv("validation_evidence.csv", index=False)

print("\nAdditional population checks")
print("Tickets after deduplication:", tickets["ticket_id"].nunique())
print("Tickets with resolution timestamps:", mapped["ticket_id"].nunique())
print("Tickets without resolution timestamps:", tickets["resolved_at"].isna().sum())
print("Potential IVR transcripts:", int(
    tickets["customer_message"].str.startswith(
        "[IVR transcript]", na=False
    ).sum()
))

,check,result,expected,passed,status
0,Duplicate ticket IDs after deduplication,0,0,True,PASS
1,Missing first-response timestamps,0,0,True,PASS
2,Negative response durations,0,0,True,PASS
3,Tickets missing SLA targets,0,0,True,PASS
4,Resolved tickets without historical roster match,0,0,True,PASS
5,Duplicate rows in weekly report grouping,0,0,True,PASS



Additional population checks
Tickets after deduplication: 11200
Tickets with resolution timestamps: 10611
Tickets without resolution timestamps: 589
Potential IVR transcripts: 916
